# Developing Multi System Agents. Last cell demonstrates the workflow.

I synthensized down the code from previous challenges.

In [1]:
pip install google-adk[gcp]

In [2]:
import os
import ast
import requests
from google.adk import Agent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from typing import Optional

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-01-4295d9eedc89"
LOCATION = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION


In [7]:
from google.adk.tools import google_search
google_search_agent = Agent(
 name="google_search_agent",
 model="gemini-3.6-flash",
    description="Searches Google for information and answers general questions.",
 instruction="Return results based upon the users search.",
 tools=[google_search])

print("Agent created:", google_search_agent.name)

Agent created: google_search_agent


In [4]:

def get_weather(city: str) -> dict:
    """Get current weather for a given city using the Open-Meteo API.

    Args:
        city: Name of the city, such as London or Washington DC.

    Returns:
        A dictionary containing current weather information
        or an error message.
    """
    print(f"TOOL CALLED: get_weather({city})")

    try:
        # Step 1: Convert city name into coordinates
        geo_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10
        )
        geo_response.raise_for_status()

        locations = geo_response.json().get("results", [])

        if not locations:
            return {"status": "error", "message": f"City not found: {city}"}

        location = locations[0]
        latitude = location["latitude"]
        longitude = location["longitude"]

        # Step 2: Request current weather
        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": (
                    "temperature_2m,relative_humidity_2m,"
                    "apparent_temperature,precipitation,"
                    "weather_code,wind_speed_10m"
                ),
                "temperature_unit": "fahrenheit",
                "wind_speed_unit": "mph",
                "timezone": "auto"
            },
            timeout=10
        )
        weather_response.raise_for_status()

        data = weather_response.json()

        return {
            "status": "success",
            "city": location["name"],
            "region": location.get("admin1"),
            "country": location.get("country"),
            "observation_time": data["current"]["time"],
            "timezone": data["timezone"],
            "temperature_f": data["current"]["temperature_2m"],
            "feels_like_f": data["current"]["apparent_temperature"],
            "humidity_percent": data["current"]["relative_humidity_2m"],
            "precipitation_mm": data["current"]["precipitation"],
            "wind_speed_mph": data["current"]["wind_speed_10m"],
            "weather_code": data["current"]["weather_code"]
        }

    except (requests.RequestException, ValueError, KeyError) as e:
        return {"status": "error", "message": str(e)}


In [8]:

weather_agent = Agent(
    name="WeatherBot",
    model="gemini-3.6-flash",
    instruction="""  When a user asks about current weather:
    1. Identify the city they are asking about.
    2. Call the get_weather tool.
    3. Use the tool's results to answer their question.
    4. Provide temperature, conditions, humidity,
       and wind speed when relevant.

    Never invent current weather information.
    If the tool returns an error, explain the problem.
    The weather_code is a WMO numeric code; do not
    guess its description if you are uncertain.""",
    tools=[get_weather],
     description="Retrieves current weather conditions and forecasts for locations."

)
print("Agent created:", weather_agent.name)


Agent created: WeatherBot


In [11]:
root = Agent(
    name ="root_agent",
    model="gemini-3.6-flash",
    instruction="""  You are a routing agent responsible for delegating user requests
    to the appropriate specialized subagent.

    You have two subagents:

    1. google_search_agent
       - Handles general knowledge questions, factual lookups,
         current events, and requests requiring Google Search.

    2. weather_agent
       - Handles questions about current weather, temperatures,
         forecasts, and weather conditions for any location.

    Rules:
    - Delegate all weather-related requests to weather_agent.
    - Delegate all other information requests to google_search_agent.
    - Do not answer user questions yourself.
    - Do not perform searches or retrieve weather information yourself.
    - If a request requires both agents, delegate to each as needed.
    - Let the appropriate subagent handle the request and provide
      its response to the user.""",
    sub_agents=[
        weather_agent, google_search_agent
    ]
)

In [10]:
APP_NAME = "Routing"
USER_ID = "workshop_user"

session_service = InMemorySessionService()

runner = Runner(
    agent=root,
    app_name=APP_NAME,
    session_service=session_service
)

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID
)

SESSION_ID = session.id
print(f"Session created: {SESSION_ID}")


Session created: b05ab7fc-bff6-4b68-8645-ef24156ecda3


In [18]:

async def demo_agent(prompt):
    print(f"\n{'=' * 60}")
    print(f"USER: {prompt}")
    print(f"{'=' * 60}")
    session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID
    )

    session_id = session.id

    message = types.Content(
        role="user",
        parts=[types.Part(text=prompt)]
    )

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message
    ):
        # Identify which agent produced the event
        print(f"\n[AGENT: {event.author}]")

        # Show agent delegation and tool calls
        for call in event.get_function_calls():
            if call.name == "transfer_to_agent":
                print(f"  ROUTING → {call.args.get('agent_name')}")
            else:
                print(f"  TOOL CALL → {call.name}")
                print(f"  ARGUMENTS → {call.args}")

        # Show tool responses
        for response in event.get_function_responses():
            if response.name != "transfer_to_agent":
                print(f"  TOOL RESULT → {response.name}")
                print(f"  RESULT → {response.response}")

        # Display the final answer
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"\nFINAL ANSWER:\n{part.text}")


In [19]:

# Test 1: Weather routing
await demo_agent("What's the current temperature in London?")

# Test 2: Search routing
await demo_agent("Who won the 2025 Nobel Prize in Physics?")

# Test 3: Semantic routing
await demo_agent("Do I need an umbrella in Seattle right now?")

# Test 4: Multi-agent request
await demo_agent(
    "What's the weather in Paris, and who is the current president of France?"
)



USER: What's the current temperature in London?

[AGENT: root_agent]
  ROUTING → WeatherBot

[AGENT: root_agent]

[AGENT: WeatherBot]
  TOOL CALL → get_weather
  ARGUMENTS → {'city': 'London'}
TOOL CALLED: get_weather(London)

[AGENT: WeatherBot]
  TOOL RESULT → get_weather
  RESULT → {'status': 'success', 'city': 'London', 'region': 'England', 'country': 'United Kingdom', 'observation_time': '2026-10-13T20:45', 'timezone': 'Europe/London', 'temperature_f': 61.3, 'feels_like_f': 57.4, 'humidity_percent': 66, 'precipitation_mm': 0.0, 'wind_speed_mph': 9.4, 'weather_code': 0}

[AGENT: WeatherBot]

FINAL ANSWER:
The current temperature in London is **61.3°F** (feels like 57.4°F) / approximately **16.3°C**.

**Current conditions:**
- **Humidity:** 66%
- **Wind Speed:** 9.4 mph
- **Precipitation:** 0 mm

USER: Who won the 2025 Nobel Prize in Physics?

[AGENT: root_agent]
  ROUTING → google_search_agent

[AGENT: root_agent]

[AGENT: google_search_agent]

FINAL ANSWER:
The **2025 Nobel Prize